In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_auc_score, roc_curve
import joblib
import warnings

from jpeg_compression_detection.paths import (
    DATA_DIR,
    GENERATED_DATA_DIR,
    MODELS_DIR,
    FIGURES_DIR,
    display_path,
)

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

# Load the two feature tables used to train the binary classifier.
df_jfif = pd.read_csv(DATA_DIR / 'results_jfif.csv')
df_ppm = pd.read_csv(DATA_DIR / 'results_ppm.csv')

df_jfif['label'] = 'jfif'
df_ppm['label'] = 'ppm'
df = pd.concat([df_jfif, df_ppm], ignore_index=True)

print(f"Total number of samples: {len(df)}")
print(f"Number of JFIF samples: {len(df_jfif)}")
print(f"Number of PPM samples: {len(df_ppm)}")
print(f"\nData shape: {df.shape}")

feature_columns = [
    'blockiness_0', 'blockiness_1', 'blockiness_2', 'blockiness_3',
    'blockiness_4', 'blockiness_5', 'blockiness_6', 'blockiness_7',
    'blockiness_max', 'blockiness_mean', 'blockiness_std',
    'energy_mean', 'energy_std', 'estimated_quality', 'psnr',
]

X = df[feature_columns]
y = df['label']
y_binary = (y == 'ppm').astype(int)

# Reproduce the training notebook's deterministic 64/16/20 split.
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y_binary, test_size=0.2, random_state=42,
    stratify=y_binary, shuffle=True,
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.2, random_state=42,
    stratify=y_temp, shuffle=True,
)

X_test.head()


In [ ]:
loaded_model = joblib.load(MODELS_DIR / 'best_classification_model.pkl')
loaded_scaler = joblib.load(MODELS_DIR / 'scaler.pkl')
loaded_feature_columns = joblib.load(MODELS_DIR / 'feature_columns.pkl')

print("Model loaded successfully.")
print(f"Model type: {type(loaded_model).__name__}")
print(f"Number of features: {len(loaded_feature_columns)}")

# Predict the first sample from the deterministic test split.
test_sample = X_test.iloc[0:1]
print(f"\nTest sample file: {df.iloc[X_test.index[0]]['filename']}")
print(f"True label: {'PPM' if y_test.iloc[0] == 1 else 'JFIF'}")
test_sample_scaled = loaded_scaler.transform(test_sample)

prediction = loaded_model.predict(test_sample_scaled)[0]
probabilities = loaded_model.predict_proba(test_sample_scaled)[0]

print(f"\nPrediction: {'PPM' if prediction == 1 else 'JFIF'}")
print(f"Probabilities: JFIF={probabilities[0]*100:.2f}%, PPM={probabilities[1]*100:.2f}%")


# Test the model on other datasets


## Load an optional feature table


In [37]:
# This external evaluation table is optional and is not distributed with the repository.
optional_csv_path = GENERATED_DATA_DIR / 'results_iFakeFaceDB.csv'
df_new = None
X_new = None

if not optional_csv_path.exists():
    print(f"Skipping optional dataset because it is not available: {display_path(optional_csv_path)}")
else:
    df_new = pd.read_csv(optional_csv_path)
    X_new = df_new[loaded_feature_columns]
    display(df_new.head())


,filename,extension,blockiness_0,blockiness_1,blockiness_2,blockiness_3,blockiness_4,blockiness_5,blockiness_6,blockiness_7,blockiness_max,blockiness_mean,blockiness_std,energy_mean,energy_std,estimated_quality,psnr
0,8,.jpg,0.992666,0.992800,1.004289,0.990242,1.005639,1.021728,1.020389,0.971611,1.021728,0.999920,0.015592,238.106247,175.519073,95,62.291457
1,9,.jpg,1.022676,0.997655,0.998614,0.968157,1.012759,1.005373,1.000040,0.994788,1.022676,1.000008,0.014809,379.096313,271.296814,95,61.224648
2,10,.jpg,0.975634,0.968364,0.986628,0.978255,1.024780,1.046203,1.001363,1.020064,1.046203,1.000161,0.025979,286.234802,231.108597,95,61.827569
3,11,.jpg,0.978805,0.952967,1.003566,1.031920,1.008710,0.973042,1.014269,1.038680,1.038680,1.000244,0.027827,212.092270,208.680054,95,62.361964
4,14,.jpg,0.989128,1.030054,1.006470,0.982038,0.955791,0.996481,1.020729,1.020462,1.030054,1.000144,0.022916,344.745850,226.218231,95,62.210988


# Test one sample


In [ ]:
if X_new is None:
    print("Skipping the single-sample test because the optional dataset is unavailable.")
else:
    test_sample = X_new.iloc[0:1]
    print(f"\nTest sample file: {df_new.iloc[X_new.index[0]]['filename']}")
    test_sample_scaled = loaded_scaler.transform(test_sample)

    prediction = loaded_model.predict(test_sample_scaled)[0]
    probabilities = loaded_model.predict_proba(test_sample_scaled)[0]

    print(f"\nPrediction: {'PPM' if prediction == 1 else 'JFIF'}")
    print(f"Probabilities: JFIF={probabilities[0]*100:.2f}%, PPM={probabilities[1]*100:.2f}%")


# Test the complete optional dataset


In [ ]:
if X_new is None:
    print("Skipping the full-dataset test because the optional dataset is unavailable.")
else:
    print(f"Total number of samples: {len(df_new)}")
    missing_count = X_new.isnull().any(axis=1).sum()
    print(f"Rows containing missing values: {missing_count}")

    valid_indices = X_new.dropna().index
    X_new_clean = X_new.loc[valid_indices]
    df_new_clean = df_new.loc[valid_indices].copy()
    print(f"Samples after removing missing values: {len(X_new_clean)}")

    X_new_scaled = loaded_scaler.transform(X_new_clean)
    predictions = loaded_model.predict(X_new_scaled)
    probabilities = loaded_model.predict_proba(X_new_scaled)

    df_new_clean['prediction'] = ['PPM' if pred == 1 else 'JFIF' for pred in predictions]
    df_new_clean['probability_JFIF'] = probabilities[:, 0] * 100
    df_new_clean['probability_PPM'] = probabilities[:, 1] * 100

    print("\nPrediction distribution:")
    print(df_new_clean['prediction'].value_counts())
    print("\nPercentage distribution:")
    print(df_new_clean['prediction'].value_counts(normalize=True) * 100)

    print(f"\n{'='*80}")
    print("Samples classified as PPM (uncompressed):")
    print('='*80)
    ppm_samples = df_new_clean[df_new_clean['prediction'] == 'PPM']
    for _, row in ppm_samples.iterrows():
        print(
            f"{row['filename']}{row['extension']} - estimated quality: "
            f"{row['estimated_quality']}, confidence: {row['probability_PPM']:.2f}%"
        )

    # Optional export:
    # output_path = GENERATED_DATA_DIR / 'predictions_optional_dataset.csv'
    # df_new_clean.to_csv(output_path, index=False)


# Analyze one image

Select an image from the data/examples directory to compute its indicators and run the classifier.


In [31]:
from jpeg_compression_detection import detect_jpeg_compression

def check_image(image_path):
    """Extract indicators and classify one image as JFIF or PPM."""
    if not image_path.exists():
        print(f"Skipping optional example because it is not available: {display_path(image_path)}")
        return

    print(f"Analyzing file: {display_path(image_path)}")
    print("-" * 80)

    try:
        result = detect_jpeg_compression(image_path)
        if not isinstance(result, dict):
            print("The image indicators could not be computed.")
            return

        feature_dict = {column: result.get(column, 0) for column in loaded_feature_columns}
        X_single = pd.DataFrame([feature_dict])

        if X_single.isnull().any().any():
            print("\nMissing values in the indicators:")
            print(X_single.isnull().sum()[X_single.isnull().sum() > 0])
            return

        X_single_scaled = loaded_scaler.transform(X_single)
        prediction = loaded_model.predict(X_single_scaled)[0]
        probabilities = loaded_model.predict_proba(X_single_scaled)[0]

        print("\n" + "-" * 80)
        print("Prediction result:")
        print("-" * 80)
        print(f"Format: {'PPM (uncompressed)' if prediction == 1 else 'JFIF (JPEG compression)'}")
        print("\nProbabilities:")
        print(f"  JFIF: {probabilities[0]*100:6.2f}%")
        print(f"  PPM:  {probabilities[1]*100:6.2f}%")
        print(f"\nConfidence: {max(probabilities) * 100:.1f}%")

    except Exception as error:
        print(f"Error: {error}")
        import traceback
        traceback.print_exc()


In [ ]:
example_paths = [
    DATA_DIR / 'examples' / 'test-500-700q80-1do1-doPNG-224-224.png',
    DATA_DIR / 'examples' / 'fakeface.jpg',
    DATA_DIR / 'examples' / 'fakeface-100x100.png',
    DATA_DIR / 'examples' / 'fakeface-500x500.png',
    DATA_DIR / 'examples' / 'fakeface-250x250.png',
    DATA_DIR / 'examples' / 'test-500-700q60q100-1000x1500.png',
]

for example_path in example_paths:
    check_image(example_path)
